In [ ]:
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
import warnings
from pint.errors import UnitStrippedWarning
warnings.simplefilter("ignore", UnitStrippedWarning)
import xarray as xr

import prism

from imagematerials.model import (
    MaterialIntensities,
    SharesInflowStocks,
    GenericStocks
)
from imagematerials.factory import ModelFactory
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.concepts import create_h2_graph
from imagematerials.util import resolve_circular_economy_scenario

# TODO absolute path of file "preprocessing.py" ? current solution can differ depending on IDE used
path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_base = Path(path_base, "data", "raw")
ce_scenario_path = Path(path_base, "circular_economy_scenarios")

graph_h2 = create_h2_graph()

# Run Model

## hydrogen production

In [ ]:
scenario_list = {
                # "SSP2_baseline_base":    ("SSP2_baseline", "base"),
                # "SSP2_baseline_full_ce": ("SSP2_baseline", "full_ce"),
                "SSP2_baseline":    ("SSP2_baseline", "base"),
                "SSP2_mitigation":   ("SSP2_mitigation", "base"),
                }

time_start = 2000
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
    circular_economy_scenario_dir = resolve_circular_economy_scenario(ce_scenario_path, circular_scen)

    h2_sector = get_preprocessing_data("hydrogen", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False,
                                    resource_efficiency_flags_file=circular_economy_scenario_dir) 
    
    factory = ModelFactory(
    h2_sector, complete_timeline
    ).add(GenericStocks, ["hydrogen"],
    ).add(MaterialIntensities, ["hydrogen"],
    )
    model = factory.finish()
    
    model.simulate(simulation_timeline)

    all_output[scen_id] = model

    print(f"Finished {scen_id}")


## hydrogen production + H2P (electricity power generation)

In [ ]:
scenario_list = {
                # "SSP2_baseline_base":    ("SSP2_baseline", "base"),
                # "SSP2_baseline_full_ce": ("SSP2_baseline", "full_ce"),
                "SSP2_baseline":    ("SSP2_baseline", "base"),
                "SSP2_mitigation":   ("SSP2_mitigation", "base"),
                }

time_start = 2000
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
    circular_economy_scenario_dir = resolve_circular_economy_scenario(ce_scenario_path, circular_scen)

    h2_sector = get_preprocessing_data("hydrogen", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False,
                                    resource_efficiency_flags_file=circular_economy_scenario_dir)

    elc_sector = get_preprocessing_data("electricity", path_base,
                                            climate_policy_scenario_dir, 
                                            circular_economy_scenario_dir, cache = False,
                                        resource_efficiency_flags_file=circular_economy_scenario_dir)
    
    factory = ModelFactory(
    [h2_sector, *elc_sector], complete_timeline
    ).add(GenericStocks, ["hydrogen"],
    ).add(SharesInflowStocks, ["elc_gen"],
    ).add(MaterialIntensities, ["hydrogen", "elc_gen"],
    )
    model = factory.finish()
    
    model.simulate(simulation_timeline)

    all_output[scen_id] = model

    print(f"Finished {scen_id}")


# Example Plots

In [ ]:
TYPE_COLORS = {
    "CBF": "#6b6b6b",
    "CBF+CCS": "#b08968",
    "electrolysis_AEL": "#1f77b4",
    "electrolysis_PEMEL": "#2ca02c",
    "electrolysis_SOEL": "#ff7f0e",
}

unit ="GW"
model = all_output["SSP2_baseline"]
hydrogen_b = model.hydrogen["inflow"].to_array().sel(time=slice(2010, 2100)).sum(["Region"]).pint.to(unit)

model = all_output["SSP2_mitigation"]
hydrogen_m = model.hydrogen["inflow"].to_array().sel(time=slice(2010, 2100)).sum(["Region"]).pint.to(unit)

fig, ax = plt.subplots(1,2,figsize=(15,6))

for t in hydrogen_b.Type.values:

    ax[0].plot(hydrogen_b.time, hydrogen_b.sel(Type=t), label=str(t), color=TYPE_COLORS.get(t))
    ax[1].plot(hydrogen_m.time, hydrogen_m.sel(Type=t), color=TYPE_COLORS.get(t))

    
ax[0].legend(loc="upper left")

ax[0].set_xlabel("Year")
ax[1].set_xlabel("Year")
ax[0].set_ylabel(f"inflow ({unit})")
ax[1].set_ylabel(f"inflow ({unit})")
ax[0].set_title("SSP2 baseline")
ax[1].set_title("SSP2 mitigation")

ax[0].grid(alpha=0.3)
ax[1].grid(alpha=0.3)


In [ ]:
COLOR_MATERIALS = {
    "copper": "#B87333",
    "nickel": "#1EE04E",
    "cobalt": "#0047AB",
    "iridium": "#DA70D6",
    "zirconium": "#54A4F3",
    "platinum": "#F1A307",
    "palladium": "#811818",
    "yttrium": "#0AEEC8",
}
unit = "kilotonne"
materials = ["cobalt", "iridium", "platinum", "palladium", "yttrium", "zirconium"]

model = all_output["SSP2_baseline"]
h2p = model.elc_gen["inflow_materials"].to_array().sel(Type=["H2P_PEMFC", "H2P_SOFC", "H2P_PAFC", "H2CHP_PEMFC", "H2CHP_SOFC", "H2CHP_PAFC"], time=slice(2010, 2100)).sum(["Type","Region"]).pint.to(unit) #time=slice(2020, 2060)
hp = model.hydrogen["inflow_materials"].to_array().sel(time=slice(2010, 2100)).sum(["Type","Region"]).pint.to(unit)
hp, h2p = xr.align(
    hp.pint.dequantify(),
    h2p.pint.dequantify(),
    join="outer",
    fill_value=0,
)
hydrogen_b = (hp + h2p).pint.quantify(unit)

model = all_output["SSP2_mitigation"]
h2p = model.elc_gen["inflow_materials"].to_array().sel(Type=["H2P_PEMFC", "H2P_SOFC", "H2P_PAFC", "H2CHP_PEMFC", "H2CHP_SOFC", "H2CHP_PAFC"], time=slice(2010, 2100)).sum(["Type","Region"]).pint.to(unit) #time=slice(2020, 2060)
hp = model.hydrogen["inflow_materials"].to_array().sel(time=slice(2010, 2100)).sum(["Type","Region"]).pint.to(unit)
hp, h2p = xr.align(
    hp.pint.dequantify(),
    h2p.pint.dequantify(),
    join="outer",
    fill_value=0,
)
hydrogen_m = (hp + h2p).pint.quantify(unit)

fig, ax = plt.subplots(figsize=(11,8))

for mat in materials:
    color = COLOR_MATERIALS.get(mat)

    if mat in hydrogen_b.material.values:
        ax.plot(hydrogen_b.time, hydrogen_b.sel(material=mat), label=str(mat), color=color)
        ax.plot(hydrogen_m.time, hydrogen_m.sel(material=mat), color=color, linestyle="--")
    
ax.legend(loc="upper left")

ax.set_xlabel("Year")
ax.set_ylabel(f"material inflow ({unit})")
ax.legend()
ax.grid(alpha=0.3)